# SAC arrival_v2 — history k=4→8 ablation on single_cross_s0 (seed=42, 1M, vanilla)

**Pre-context（commits `01b78ad` / `3d20e86`）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.6 + §7.7 已闭环。`single_cross_s0` 是 8 个 strict-control cell 中唯一 catastrophic FAIL 的：

| 已闭环 cell | algo | sensor | history | final | OOB | 5/5 Gate | 解释 |
|---|---|---|---|---:|---:|---|---|
| §7.1 single_cross_s1_k4 | vanilla | s1 | k=4 | 0.900 | 0.100 | PASS（borderline） | s1 双点 sensor 信息够 |
| §7.6.4 single_cross_s0_k4 | vanilla | s0 | k=4 | 0.100 | 0.667 | FAIL（3/5） | s0 单点 + 短 history，信息不足 |
| §7.7 single_cross_s0_k4 + asym | sac_asym | s0 | k=4 | 0.167 | 0.633 | **FAIL（3/5）**；mean −5× | critic 信息升级，但 actor 端没新信息 → 行为风格变保守，但 task-level 反而更差 |

**§7.7 的 negative finding 把 `single_cross_s0` 的瓶颈从「critic estimation accuracy」重定位到「actor-side information access」**。本 notebook 测这个重定位后的新假设：

> 给 actor 更长的时序窗口（仍然只用 DVL 单点 sensor），让 actor 能从 single-point 时序信号里**反演**出涡街相位 / 局部 hull-integral flow 的代理量。

**本 notebook 任务（pure vanilla + history k=4→8 单变量 ablation）**：把 §7.6.4 的 vanilla SAC 配置**唯一变量**换成 `--history-length 8`，其它**全部不动**：

| 维度 | §7.6.4 baseline (vanilla, k=4) | 本 notebook |
|---|---|---|
| `--history-length` | 4 | **8** ← 唯一变量 |
| algorithm | vanilla SAC | vanilla SAC（**不**带 AsymCritic / LN / UTD） |
| sensor layout | s0 (DVL-only) | s0 (DVL-only)，仍然 deployment-realistic |
| reward | arrival_v2 | arrival_v2 |
| flow U / target | 1.5 / 1.5 | 1.5 / 1.5 |
| seed | 42 | 42 |
| total_steps | 1M | 1M |
| num_envs | 6 | 6 |
| benchmark | `single_u15_cross_tgt15` | 同 |
| obs_dim | 10×4 + 8(context) = 48 | **10×8 + 8(context) = 88** |

物理意义：每个 control step 是 ~0.5 s（dt × eval_action_repeat）。k=4 历史窗 ~2 s，远低于涡街周期 10–20 s（仅覆盖 10–20%）。k=8 历史窗 ~4 s，覆盖 20–40% 涡街周期 — 已足以让网络从单点 DVL 的时序节拍中读出主导脉动相位（即 critic 在 §7.7 通过 privileged `[u_eq, v_eq]` 看到的信息的时序代理）。

**为什么 k=8 而不是 k=12 / k=16？** 阶梯式逼近：先验证「actor temporal info 有效」这个假设；如果 k=8 PASS → 后续可扫 k=12 / k=16 看是否单调改善；如果 k=8 FAIL → 强证据指向 s0 + 物理 sensor 信息存在 information-theoretic ceiling，下一步走 §8 P3 #10 物理机制深挖。

**Gate**（与 §7 / §7.6 / §7.7 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Single seed (=42)**。**Exploratory**，不是 thesis-grade。判读规则（驱动 §8 P1 后续展开）：
- **PASS**（final ≥ 0.85, OOB ≤ 0.10）→ "s0 + 更长 history 闭合 sensor 信息瓶颈"。下一步：① 短附录 k=12 / k=16 scan ② multi-seed 复现写 §7.8。
- **STRONG-PARTIAL**（final ∈ [0.5, 0.85)，OOB ≤ 0.30）→ history 部分缓解 information bottleneck，但还不够。下一步：k=12 / k=16 单调性。
- **MARGINAL**（final ∈ [0.2, 0.5) 且 mean_success 比 §7.6.4 / §7.7 明显更高）→ 有效但远未充分；启动 §8 P3 #10 物理机制深挖。
- **NO-EFFECT**（final < 0.2 且 mean_success 与 §7.6.4 / §7.7 同量级）→ 强证据：s0 在 cross 几何下 information-theoretic ceiling 不可破。把 §7.7 的 negative finding 升格为 thesis 的 honest conclusion。

**输出根（与 §7.6.4 / §7.7 互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42/`

**总预算**：~2.5h L4（1 Colab Pro+ session；obs_dim 88 vs 48 略增 forward cost，但 hidden_dim 仍 256，主要瓶颈是 env step 而非网络）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout），与 §7 / §7.6 / §7.7 一致。


## 0. GPU sanity


In [1]:
!nvidia-smi | head -10


Mon May 18 03:55:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |


## 1. Mount Drive + cwd


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. Config — single phase（vanilla SAC + history k=4→8，与 §7.6.4 仅差一个 flag value）


In [3]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.6.4 baseline 严格一致，除 history_length 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 8                      # ← 唯一与 §7.6.4 / §7.7 不同的值 (was 4)
TARGET_SPEED = 1.5
SEED = 42

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — 与 §7.6.4 一致，避免 confound
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow file（与 §7.1 / §7.6.4 / §7.7 严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 k=8 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines（事后对比，全部用 final_eval.json 实读）
X_K4_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')      # §7.6.4
X_K4_ASYM_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42')    # §7.7
X_S1_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')      # §7.1

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}        ← 唯一与 §7.6.4 / §7.7 不同 (was 4)')
print(f'SEED                  = {SEED}')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 10 * {HISTORY_LENGTH} + 8 (arrival_v2 context) = {10 * HISTORY_LENGTH + 8}')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§7.6.4 k=4 baseline   : {X_K4_BASELINE_ROOT}')
print(f'§7.7 k=4 asym baseline: {X_K4_ASYM_BASELINE_ROOT}')
print(f'§7.1 s1_k4 reference  : {X_S1_BASELINE_ROOT}')


PROBE_LAYOUT          = s0
OBJECTIVE             = arrival_v2
HISTORY_LENGTH        = 8        ← 唯一与 §7.6.4 / §7.7 不同 (was 4)
SEED                  = 42
NUM_ENVS              = 6
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM         = False
UPDATES_PER_STEP      = 1
DROPOUT_RATE          = 0.0

benchmark             : single_u15_cross_tgt15
geometry              : cross_stream
total_steps           : 1,000,000
expected obs_dim      : 10 * 8 + 8 (arrival_v2 context) = 88
run_root              : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42
§7.6.4 k=4 baseline   : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42
§7.7 k=4 asym baseline: experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42
§7.1 s1_k4 reference  : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42


## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [4]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


[OK] flow file: wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy  (230.4 MB)


In [5]:
!python -u -m scripts.validate_arrival_v2_candidate



[undiscounted]
fast_success       144.473
slow_success       142.023
unsafe_success      91.048
timeout_near      -149.800
timeout_far       -229.800
late_oob          -274.500
fast_oob          -280.325
mid_oob           -319.450

[discounted_gamma_0.995]
fast_success        88.578
slow_success        54.683
unsafe_success      34.857
timeout_near       -39.480
timeout_far        -58.269
late_oob          -103.800
mid_oob           -190.006
fast_oob          -212.555

[discounted_shortcut] safe=69.330 risky=62.361

PASS: arrival_v2 candidate pre-integration gates passed.


In [6]:
!python -u -m pytest tests/test_reward_objective.py -q


...............                                                          [100%]
15 passed in 22.04s


In [7]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


[OK] manifest ready: benchmarks/single_u15_cross_tgt15.json


In [8]:
# 检查 3 个对比 baseline 是否就位 — 不影响训练，仅 §5 ablation diff 用
for label, root, ref_final, ref_oob in [
    ('§7.6.4 vanilla_s0_k4', X_K4_BASELINE_ROOT, 0.100, 0.667),
    ('§7.7 sac_asym_s0_k4 ', X_K4_ASYM_BASELINE_ROOT, 0.167, 0.633),
    ('§7.1 vanilla_s1_k4  ', X_S1_BASELINE_ROOT, 0.900, 0.100),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        print(f'[WARN] {label}: {fp} 不存在 (后续 §5 diff 会回退到 report 转载值)')


[OK] §7.6.4 vanilla_s0_k4: final=0.1000  oob=0.6667  counts={'out_of_bounds': 20, 'timeout': 7, 'goal': 3}  ✓
[OK] §7.7 sac_asym_s0_k4 : final=0.1667  oob=0.6333  counts={'out_of_bounds': 19, 'goal': 5, 'timeout': 6}  ✓
[OK] §7.1 vanilla_s1_k4  : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓


## 4. Train — single_cross_s0 + history k=8 (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


[state] X env_step = 0 / target 1,000,000
[train] X fresh start -> 1,000,000
[train] episode=5 step=642 return=-251.36 success=False time=17.5s geometry=cross_stream history=8
[train] episode=10 step=1122 return=-414.93 success=False time=93.3s geometry=cross_stream history=8
[train] episode=15 step=1326 return=-520.75 success=False time=110.4s geometry=cross_stream history=8
[train] episode=20 step=1878 return=-403.95 success=False time=61.3s geometry=cross_stream history=8
[train] episode=25 step=2712 return=-340.63 success=False time=68.7s geometry=cross_stream history=8
[train] episode=30 step=3330 return=-411.30 success=False time=38.1s geometry=cross_stream history=8
[train] episode=35 step=3954 return=-267.16 success=False time=8.0s geometry=cross_stream history=8
[train] episode=40 step=4350 return=-264.62 success=False time=11.7s geometry=cross_stream history=8
[train] episode=45 step=5046 return=-429.74 success=False time=104.3s geometry=cross_stream history=8 | q1=511.342 ac

## 5. Summary + gate + ablation diff vs §7.6.4 / §7.7 / §7.1


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    # Parse train_config.txt for history_length confirmation (trainer_state.json 不一定 dump 这个)
    history_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            if ln.strip().startswith('history_length='):
                history_from_config = ln.strip().split('=', 1)[1]
                break

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=8 / vanilla / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   (39 evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}   (expect 10*8+8=88)")
    print(f"  history_length        : {history_from_config}   (from train_config.txt)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_vanilla',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_K8_ABLATION',
    'single_cross_s0_k8_ablation_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


SINGLE_CROSS_S0_K8_ABLATION  (arrival_v2 / s0 / k=8 / vanilla / 1,000,000 steps)
----------------------------------------------------------------------------------------------------
  final_success_rate    : 0.9000   gate >= 0.85
  peak_success_rate     : 0.9000   @ 475,002
  last100k_mean_success : 0.9000   gate >= 0.8100
  full-traj mean        : 0.6359   (39 evals)
  n_evals_with_success  : 37 / 39
  final_oob_rate        : 0.1000   gate <= 0.10
  obs_dim               : 96   (expect 10*8+8=88)
  history_length        : 8   (from train_config.txt)
  context_obs           : True
  timeout_bootstrap     : terminal
  termination           : {'goal': 27, 'out_of_bounds': 3}

[last 16 eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   600000           0.666667    20.563137         13.560604    80.796667             0.799365
   625002           0.866667    67.911367         11.459453    87.610000             0.768013
   650004      

## 6. Ablation verdict — k=8 vs k=4(vanilla) vs k=4(asym) vs k=4(s1 ref)


In [ ]:
def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None, label: str = ''):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'source': 'report'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
    return out

print('=' * 100)
print('SINGLE_CROSS — history k=4→8 ablation verdict')
print('-' * 100)

# 当前 k=8 结果
k8_final = float(x_summary['final_success_rate'])
k8_oob = float(x_summary['final_oob_rate'])
k8_peak = float(x_summary['peak_success_rate'])
k8_mean = float(x_summary.get('mean_success_full_trajectory', 0.0))
k8_n_succ = int(x_summary.get('n_evals_with_success', 0))
k8_n_tot = int(x_summary.get('n_evals_total', 0))

# 三个 baseline
b_k4_v = read_baseline(X_K4_BASELINE_ROOT,      0.100, 0.667, 0.221, '§7.6.4 vanilla k=4')
b_k4_a = read_baseline(X_K4_ASYM_BASELINE_ROOT, 0.167, 0.633, 0.044, '§7.7 sac_asym k=4')
b_s1   = read_baseline(X_S1_BASELINE_ROOT,      0.900, 0.100, None,  '§7.1 vanilla s1 k=4')

print()
print(f'{"config":<42}{"final":>10}{"mean":>10}{"oob":>10}{"source":>14}')
print('-' * 100)
print(f'{"vanilla SAC, s1_k4 (§7.1, upper ref)":<42}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{b_s1["source"]:>14}')
print(f'{"vanilla SAC, s0_k4 (§7.6.4 baseline)":<42}{b_k4_v["final"]:>10.4f}{(b_k4_v["mean"] or float("nan")):>10.4f}{b_k4_v["oob"]:>10.4f}{b_k4_v["source"]:>14}')
print(f'{"sac_asym, s0_k4 (§7.7 negative ref)":<42}{b_k4_a["final"]:>10.4f}{(b_k4_a["mean"] or float("nan")):>10.4f}{b_k4_a["oob"]:>10.4f}{b_k4_a["source"]:>14}')
print(f'{"vanilla SAC, s0_k8 (this run)":<42}{k8_final:>10.4f}{k8_mean:>10.4f}{k8_oob:>10.4f}{"on-disk":>14}')
print('=' * 100)

# Δ 行
print()
print(f'{"contrast":<60}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}')
print('-' * 100)
print(f'{"k=8 vs k=4 (vanilla, §7.6.4) — main hypothesis":<60}'
      f'{k8_final - b_k4_v["final"]:>+12.4f}'
      f'{(k8_mean - (b_k4_v["mean"] or 0)):>+12.4f}'
      f'{k8_oob - b_k4_v["oob"]:>+12.4f}')
print(f'{"k=8 vs k=4+asym (§7.7) — vs critic-info upgrade":<60}'
      f'{k8_final - b_k4_a["final"]:>+12.4f}'
      f'{(k8_mean - (b_k4_a["mean"] or 0)):>+12.4f}'
      f'{k8_oob - b_k4_a["oob"]:>+12.4f}')
print(f'{"k=8 vs k=4 (s1 ref, §7.1) — remaining gap to upper":<60}'
      f'{k8_final - b_s1["final"]:>+12.4f}'
      f'{"N/A":>12}'
      f'{k8_oob - b_s1["oob"]:>+12.4f}')
print('=' * 100)
print()
print(f'k=8 evals_with_success: {k8_n_succ} / {k8_n_tot}  (vanilla k=4 was 35/39, asym k=4 was 19/39)')

# 判读 — 4 档（PASS / STRONG-PARTIAL / MARGINAL / NO-EFFECT）
if k8_final >= PASS_FINAL_SUCCESS and k8_oob <= PASS_OOB_RATE:
    verdict = 'PASS — s0 + 更长 history 闭合 sensor 信息瓶颈；下一步：k=12/16 单调扫 + multi-seed → §7.8'
elif k8_final >= 0.50 and k8_oob <= 0.30:
    verdict = 'STRONG-PARTIAL — history 部分缓解 information bottleneck，未达 gate；下一步：k=12/16 单调性扫'
elif k8_final >= 0.20 and (k8_mean > (b_k4_v["mean"] or 0) * 1.5 or k8_mean > (b_k4_a["mean"] or 0) * 1.5):
    verdict = 'MARGINAL — history 有效但远未充分；启动 §8 P3 #10 物理机制深挖（OOB 时间分布 + 涡街相位耦合）'
else:
    verdict = 'NO-EFFECT — k=8 与 k=4 同量级；强证据指向 s0 在 cross 几何下 information-theoretic ceiling；把 §7.7 升格为 thesis honest conclusion'

print()
print(f'>>> verdict: {verdict}')

# 落盘 ablation summary
ablation_out = {
    'experiment': 'arrival_v2_s0_cross_k8_ablation',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_vanilla',
    'cli_diff_vs_§7.6.4_baseline': '--history-length 4 → 8',
    'results': {
        'sac_vanilla_s0_k8_thisrun': {
            'final': k8_final, 'mean': k8_mean, 'oob': k8_oob, 'peak': k8_peak,
            'n_evals_with_success': k8_n_succ, 'n_evals_total': k8_n_tot,
        },
        'sac_vanilla_s0_k4_baseline_§7.6.4': b_k4_v,
        'sac_asym_s0_k4_baseline_§7.7': b_k4_a,
        'sac_vanilla_s1_k4_reference_§7.1': b_s1,
    },
    'delta_vs_§7.6.4_vanilla_k4': {
        'final_pp': round((k8_final - b_k4_v['final']) * 100, 2),
        'mean_pp': round((k8_mean - (b_k4_v['mean'] or 0)) * 100, 2),
        'oob_pp': round((k8_oob - b_k4_v['oob']) * 100, 2),
    },
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_k8_ablation_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(ablation_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')


SINGLE_CROSS — history k=4→8 ablation verdict
----------------------------------------------------------------------------------------------------

config                                         final      mean       oob        source
----------------------------------------------------------------------------------------------------
vanilla SAC, s1_k4 (§7.1, upper ref)          0.9000    0.4966    0.1000       on-disk
vanilla SAC, s0_k4 (§7.6.4 baseline)          0.1000    0.2205    0.6667       on-disk
sac_asym, s0_k4 (§7.7 negative ref)           0.1667    0.0436    0.6333       on-disk
vanilla SAC, s0_k8 (this run)                 0.9000    0.6359    0.1000       on-disk

contrast                                                         Δ final      Δ mean       Δ oob
----------------------------------------------------------------------------------------------------
k=8 vs k=4 (vanilla, §7.6.4) — main hypothesis                   +0.8000     +0.4154     -0.5667
k=8 vs k=4+asym (§7.